# 05a Registration gate: FPFH + RANSAC + ICP on 9 PC2Model samples

**Question.** Does the classical registration from notebook 02 (voxel downsample, FPFH features, RANSAC, point-to-plane
ICP) recover PC2Model's ground-truth transform on 9 samples: 3 houses, 3 simulated indoor scenes and 3 real indoor scans?
This is the registration half of the roadmap's Phase 0.5 gate. The other halves (perturbed model, hand check) are 05b and later.

**Slice.** Per category (house, `indoor_modelling` = simulated indoor, `ISPRS_indoor_modeling_benchmark` = real indoor):
the smallest sample with a ground-truth scale different from 1, the smallest sample with scale 1, then the next smallest
remaining one (all by compressed size, so the downloads stay small). `scripts/pick_gate_samples.py` made the choice
before this notebook ran and wrote `outputs/05a/gate_samples.csv`.

**Thresholds, fixed before any result.** Verbatim from `docs/roadmap.md`. They are judgement calls, not taken from a standard.

- Registration success for one sample, after ICP: rotation error <= 2 degrees, translation error <= 5 cm, and scale error
  <= 1% on scaled samples.
- The registration part of the gate passes if at least 6 of the 9 samples succeed.
- Perturbed CRAS model (a later notebook): every injected change beyond tolerance gets the right verdict, and at most 10%
  of untouched elements are falsely flagged shifted or missing.
- Hand check of 20 random real CRAS elements (a later notebook): at least 80% agree with the verdict.

Only the first two bullets are tested here; the other two belong to later notebooks.

**Decisions made before running** (also judgement calls).

1. *Translation error* is the **centroid error**: the distance between where the estimate and the ground truth put the
   scan's centroid. The difference of the two translation vectors, `|dt|`, is shown for context only, because a small
   rotation error times the scan's distance from its own origin (hundreds of metres here) dominates it.
2. *A sample counts on the draw picked by ICP fitness* among seeds 0 to 4 (the highest fitness; the ground truth is not
   used to pick). Notebook 02 showed that RANSAC is a lottery on a real room and that the seed declared in advance can be a
   failure, so one seed per sample would mostly measure luck. Seed 0's result is shown too, as context only.
3. *Scale.* On a sample whose ground-truth scale is not 1 (`|s - 1| > 0.001`), the scan is first multiplied by 02's
   RMS-radius estimate, then the rigid pipeline runs, and the transforms are composed (`T = T_rigid * diag(s, s, s, 1)`).
   On a sample at scale 1 the pipeline runs rigidly with scale 1, a prior that holds for real scans in metres. This uses
   the benchmark's scaled / unscaled flag, which a real run would not have. The scale criterion applies to scaled samples only.
4. The voxel fraction (0.008 of the model's bounding-box diagonal), the five seeds, the feature radii and the RANSAC and ICP
   settings are 02's, unchanged. They were tuned on UVigo's ground truth in 02, which is one of the nine samples here.

In [1]:
import csv
import gc
import io
import resource
import time
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import open3d as o3d
from remotezip import RemoteZip

from scan_vs_bim.fetch import extract_zip_member, http_range_reader
from scan_vs_bim.load import load_e57_chunked
from scan_vs_bim.pc2model import URL as PC2MODEL_URL, Sample, parse_transform, pick_gate_samples, scale_of

reg = o3d.pipelines.registration
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "pc2model"
OUT = ROOT / "outputs" / "05a"
OUT.mkdir(parents=True, exist_ok=True)
t_notebook = time.perf_counter()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ---- Every knob of this notebook is in this cell. ---------------------------------------------------------------------------------

# Gate thresholds from docs/roadmap.md (judgement calls, not from a standard); see the first cell.
GATE_ROT_DEG, GATE_TRANS_M, GATE_SCALE_REL = 2.0, 0.05, 0.01
GATE_MIN_SUCCESS = 6   # of the 9 samples

# A sample is "scaled" when its ground-truth scale differs from 1 by more than this.
SCALED_TOL = 1e-3

# Seeds. SEED: model-surface sampling and the scan subsample. SEED_SWEEP: the RANSAC draws, one per seed (reseeded
# before every RANSAC run); the draw with the highest ICP fitness counts.
SEED = 0
SEED_SWEEP = [0, 1, 2, 3, 4]

# Point budgets (02's values).
MODEL_POINTS = 500_000        # points sampled uniformly from the model mesh surface
MAX_SCAN_POINTS = 1_000_000   # at most this many scan points are loaded, read in chunks (see the memory note below)
CHUNK_POINTS = 1_000_000      # E57 points read per chunk

# Voxel size = VOXEL_FRACTION * diagonal of the model's bounding box (02's value).
VOXEL_FRACTION = 0.008

# Feature radii, as multiples of the voxel size.
NORMAL_RADIUS_X, NORMAL_MAX_NN = 2.0, 30
FPFH_RADIUS_X, FPFH_MAX_NN = 5.0, 100

# RANSAC (feature matching): max iterations, confidence, inlier distance (x voxel), edge-length check.
RANSAC_MAX_ITER, RANSAC_CONFIDENCE = 1_000_000, 0.999
RANSAC_DISTANCE_X = 1.5
RANSAC_EDGE_RATIO = 0.9
RANSAC_MUTUAL_FILTER = True

# Point-to-plane ICP: max correspondence distance (x voxel) and iterations.
ICP_DISTANCE_X, ICP_MAX_ITER = 2.0, 100

# Direction check on the ground truth: T must put the scan within this fraction of the model's diagonal (median distance),
# and inv(T) must be at least INV_FACTOR times worse. This is a direction check, not an accuracy check: it separates T from
# inv(T), which is off by orders of magnitude. The 2% limit is loose on purpose (house_18 passes at 0.406 m against a ~0.52 m
# limit because its scan covers far more than its model, so even the true T leaves a 0.406 m median residual).
DIRECTION_FRACTION, INV_FACTOR = 0.02, 10

## The nine samples

The notebook reads `outputs/05a/gate_samples.csv` (written by `scripts/pick_gate_samples.py` from the zip's central
directory and the 137 transform files; no point cloud is downloaded for the choice). As a check, the choice is recomputed
here from `outputs/pc2model_scales.csv` and compared.

In [3]:
gate_csv, scales_csv = OUT / "gate_samples.csv", ROOT / "outputs" / "pc2model_scales.csv"
for p, how in [(gate_csv, "scripts/pick_gate_samples.py"), (scales_csv, "scripts/find_scaled_sample.py")]:
    if not p.exists():
        raise FileNotFoundError(f"{p.relative_to(ROOT)} is missing. Run: uv run python {how}")
chosen = list(csv.DictReader(open(gate_csv)))
print(f"{'category':18s}{'sample':72s}{'MB':>8s}{'scale':>8s}")
for r in chosen:
    print(f"{r['label']:18s}{Path(r['sample']).name:72s}{r['compressed_mb']:>8s}{float(r['scale']):8.3f}")

# Consistency check, not a known answer: re-deriving the selection rule from the scales CSV (sizes rounded to 0.01 MB there)
# must reproduce the choice stored in gate_samples.csv. Both come from the same rule, so this catches drift, not a wrong rule.
rows = list(csv.DictReader(open(scales_csv)))
by_folder = {r["sample"]: float(r["scale"]) for r in rows}
all_samples = [Sample(r["sample"], round(float(r["compressed_mb"]) * 1e6), None) for r in rows]
categories = list(dict.fromkeys(r["category"] for r in chosen))
assert [s.folder for s in pick_gate_samples(all_samples, by_folder, categories)] == [r["sample"] for r in chosen]
assert len(chosen) == 9 and all(abs(float(r["scale"]) - by_folder[r["sample"]]) < 1e-6 for r in chosen)
print(f"\n{sum(abs(float(r['scale']) - 1) > SCALED_TOL for r in chosen)} of the 9 are scaled (ground-truth scale != 1).")

category          sample                                                                        MB   scale
house             house_23-sketchfab                                                        157.63   1.206
house             house_13-sketchfab                                                        103.40   1.000
house             house_18-sketchfab                                                        152.80   1.000
simulated indoor  indoor_modelling_4-ISPRS_indoor_modelling-UoM                             151.65   0.747
simulated indoor  indoor_modelling_3-ISPRS_indoor_modelling-UVigo                           141.40   1.000
simulated indoor  indoor_modelling_0-ISPRS_indoor_modelling-TUBS1                           189.19   0.689
real indoor       real-indoor_modelling_1-ISPRS_indoor_modelling-TUBS2                      225.44   1.258
real indoor       real-indoor_modelling_3-ISPRS_indoor_modelling-UVigo                      102.09   1.000
real indoor       real-indoor_modelli

## The method, stage by stage

Copied from notebook 02 (the stage logic stays inline until the stage is formalised). For each sample, one at a time:

1. **Fetch** the sample's three files (model OBJ, scan E57, ground-truth transform) by HTTP range requests if they are
   not in `data/pc2model/` yet. Resumable, CRC-checked (`scan_vs_bim.fetch`). Nothing else of the 10 GB zip is read.
2. **Load.** Sample the OBJ surface uniformly into a point cloud; load the scan with a seeded subsample.
3. **Scale** (scaled samples only): `s = RMS radius of the model / RMS radius of the scan`, each about its own centroid, on
   the voxel-downsampled clouds. The scan is multiplied by `s`.
4. **Voxel size** = 0.008 x the model's bounding-box diagonal. Normals and FPFH on the downsampled clouds, RANSAC over FPFH
   correspondences, then point-to-plane ICP. Five RANSAC seeds; the draw with the highest ICP fitness counts.
5. **Metrics against the ground truth `T`:** rotation error (geodesic angle, scale divided out), centroid error (the
   gate's "translation error"), `|dt|` for context, scale error, and the registration residual (median distance from
   every loaded scan point, moved by the estimate, to the model mesh; ray casting).
6. **Free** everything before the next sample.

**Memory.** On this 8 GB machine `load_e57` reads one whole scan at a time before subsampling and needs about 100 bytes per
point at its peak (1.08 GB on UVigo's single scan of 9.5M points). Most samples here are 8 to 12 scans of at most 1.7M points,
but the real indoor TUBS2 sample is one scan of 21.7M points, which would need about 2.2 GB. This notebook therefore uses
`load_e57_chunked`, which reads 1M points at a time and picks the same points (checked against `load_e57` in
`tests/test_load.py`); its peak on UVigo was 0.30 GB, and it does not grow with the scan.

In [4]:
def seed_all(seed=SEED):
    np.random.seed(seed)
    o3d.utility.random.seed(seed)


_INFOS = None   # the zip's central directory, fetched once and only if a sample has to be downloaded


def ensure_files(folder):
    # Returns (directory, retry lines printed by the downloader or None when nothing had to be fetched).
    global _INFOS
    d = DATA / Path(folder).name
    if d.exists() and all(any(d.glob(g)) for g in ("model_*.obj", "pointcloud_*_transformed.e57", "transformation_matrix_*.txt")):
        return d, None
    if _INFOS is None:
        with RemoteZip(PC2MODEL_URL) as z:
            _INFOS = z.infolist()
    reader, log = http_range_reader(PC2MODEL_URL), io.StringIO()
    with redirect_stdout(log):
        for info in _INFOS:
            if info.filename.startswith(folder + "/") and not info.is_dir():
                extract_zip_member(reader, info, d / Path(info.filename).name)
    return d, sum("retry" in line for line in log.getvalue().splitlines())


def load_sample(folder):
    d, retries = ensure_files(folder)
    mesh = o3d.io.read_triangle_mesh(str(next(d.glob("model_*.obj"))))
    T_gt = parse_transform(next(d.glob("transformation_matrix_*.txt")).read_text())
    seed_all()
    model = mesh.sample_points_uniformly(MODEL_POINTS)
    scan = load_e57_chunked(next(d.glob("pointcloud_*_transformed.e57")), MAX_SCAN_POINTS, SEED, CHUNK_POINTS)
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))
    diag = float(np.linalg.norm(mesh.get_axis_aligned_bounding_box().get_extent()))
    return dict(model=model, scan=scan, T_gt=T_gt, scene=scene, diag=diag, n_triangles=len(mesh.triangles), retries=retries)


def preprocess(pcd, voxel):
    down = pcd.voxel_down_sample(voxel)
    down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=NORMAL_RADIUS_X * voxel, max_nn=NORMAL_MAX_NN))
    fpfh = reg.compute_fpfh_feature(down, o3d.geometry.KDTreeSearchParamHybrid(radius=FPFH_RADIUS_X * voxel, max_nn=FPFH_MAX_NN))
    return down, fpfh


def register(scan, model, voxel, seeds):
    # 02's pipeline. The features are computed once; RANSAC is reseeded and rerun for each seed, then ICP refines each draw.
    src, src_f = preprocess(scan, voxel)
    tgt, tgt_f = preprocess(model, voxel)
    runs = []
    for seed in seeds:
        seed_all(seed)   # reseeded right before RANSAC, so its result does not depend on what ran before
        ransac = reg.registration_ransac_based_on_feature_matching(
            src, tgt, src_f, tgt_f, RANSAC_MUTUAL_FILTER, RANSAC_DISTANCE_X * voxel,
            reg.TransformationEstimationPointToPoint(False), 3,
            [reg.CorrespondenceCheckerBasedOnEdgeLength(RANSAC_EDGE_RATIO),
             reg.CorrespondenceCheckerBasedOnDistance(RANSAC_DISTANCE_X * voxel)],
            reg.RANSACConvergenceCriteria(RANSAC_MAX_ITER, RANSAC_CONFIDENCE))
        icp = reg.registration_icp(src, tgt, ICP_DISTANCE_X * voxel, ransac.transformation,
                                   reg.TransformationEstimationPointToPlane(),
                                   reg.ICPConvergenceCriteria(max_iteration=ICP_MAX_ITER))
        runs.append(dict(seed=seed, T=icp.transformation, icp_fitness=icp.fitness))
    return runs


def rms_radius(pcd, voxel):
    p = np.asarray(pcd.voxel_down_sample(voxel).points)
    return float(np.sqrt(((p - p.mean(0)) ** 2).sum(1).mean()))


def apply(T, pts):
    return pts @ T[:3, :3].T + T[:3, 3]


def rotation_error_deg(T_est, T_gt):
    # Geodesic angle between the two rotations; the scale is divided out of each 3x3 block first.
    A, B = T_est[:3, :3] / scale_of(T_est), T_gt[:3, :3] / scale_of(T_gt)
    return float(np.degrees(np.arccos(np.clip((np.trace(A.T @ B) - 1) / 2, -1, 1))))


def centroid_error(T_est, T_gt, pts):
    c = pts.mean(0)
    return float(np.linalg.norm(apply(T_est, c) - apply(T_gt, c)))


def median_to_model(scene, T, pts):
    return float(np.median(scene.compute_distance(o3d.core.Tensor(apply(T, pts).astype(np.float32))).numpy()))


def succeeded(rot_deg, centroid_m, scale_err, scaled):
    return rot_deg <= GATE_ROT_DEG and centroid_m <= GATE_TRANS_M and (not scaled or scale_err <= GATE_SCALE_REL)


def peak_rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9   # macOS reports bytes

In [5]:
def run_sample(r):
    t0 = time.perf_counter()
    s = load_sample(r["sample"])
    t_load = time.perf_counter() - t0
    pts, T_gt, diag = np.asarray(s["scan"].points), s["T_gt"], s["diag"]
    gt_scale = scale_of(T_gt)
    scaled = abs(gt_scale - 1) > SCALED_TOL
    voxel = VOXEL_FRACTION * diag

    # Direction check: T maps the scan onto the model, inv(T) does not.
    d_T, d_inv = median_to_model(s["scene"], T_gt, pts), median_to_model(s["scene"], np.linalg.inv(T_gt), pts)
    assert d_T < DIRECTION_FRACTION * diag and d_inv > INV_FACTOR * d_T, (r["sample"], d_T, d_inv, diag)
    print(f"  direction check {Path(r['sample']).name}: median distance under T {d_T:.3f} m, under inv(T) {d_inv:.1f} m "
          f"(limit for T {DIRECTION_FRACTION * diag:.3f} m)", flush=True)

    # Scale: the RMS-radius estimate on scaled samples (uses the benchmark's flag), the prior 1 otherwise.
    est_scale = rms_radius(s["model"], voxel) / rms_radius(s["scan"], voxel) if scaled else 1.0
    S = np.diag([est_scale, est_scale, est_scale, 1.0])
    scan_s = o3d.geometry.PointCloud(s["scan"]).transform(S)

    t1 = time.perf_counter()
    runs = register(scan_s, s["model"], voxel, SEED_SWEEP)
    t_reg = time.perf_counter() - t1
    for run in runs:
        run["T"] = run["T"] @ S   # compose: scale first, then the rigid transform
    best = max(runs, key=lambda run: run["icp_fitness"])   # ties: the earlier seed
    seed0 = runs[0]

    rot, cen = rotation_error_deg(best["T"], T_gt), centroid_error(best["T"], T_gt, pts)
    scale_err = abs(scale_of(best["T"]) - gt_scale) / gt_scale
    rot0, cen0 = rotation_error_deg(seed0["T"], T_gt), centroid_error(seed0["T"], T_gt, pts)
    row = dict(
        sample=Path(r["sample"]).name, category=r["label"], folder=r["sample"], compressed_mb=float(r["compressed_mb"]),
        scaled=scaled, gt_scale=gt_scale, est_scale=scale_of(best["T"]), scale_err_pct=100 * scale_err,
        rot_deg=rot, centroid_m=cen, dt_m=float(np.linalg.norm(best["T"][:3, 3] - T_gt[:3, 3])),
        resid_median_m=median_to_model(s["scene"], best["T"], pts), gt_resid_median_m=d_T,
        icp_fitness=best["icp_fitness"], best_seed=best["seed"], success=succeeded(rot, cen, scale_err, scaled),
        seed0_rot_deg=rot0, seed0_centroid_m=cen0,
        seed0_success=succeeded(rot0, cen0, abs(scale_of(seed0["T"]) - gt_scale) / gt_scale, scaled),
        seeds_rot_deg=";".join(f"{rotation_error_deg(run['T'], T_gt):.3f}" for run in runs),
        seeds_icp_fitness=";".join(f"{run['icp_fitness']:.3f}" for run in runs),
        n_scan_points=len(pts), model_diag_m=diag, voxel_m=voxel, n_triangles=s["n_triangles"],
        load_s=t_load, register_s=t_reg)
    note = "" if s["retries"] is None else f", downloaded ({s['retries']} retries)"
    print(f"{row['sample']}: {row['n_triangles']:,} triangles, diagonal {diag:.1f}, {len(pts):,} scan points, voxel {voxel:.3f}; "
          f"load {t_load:.0f} s{note}, register {t_reg:.0f} s; "
          f"seed {row['best_seed']} picked, {'success' if row['success'] else 'fail'}; peak RSS so far {peak_rss_gb():.2f} GB", flush=True)
    return row

## Running the nine samples

One at a time: fetch, load, register, record, free. Each finished sample is appended to the list, and
`outputs/05a/registration.csv` (read by 05b) is written when the loop ends.

In [6]:
results = []
for r in chosen:
    results.append(run_sample(r))
    gc.collect()   # drop the clouds, mesh and ray-casting scene of this sample before the next one

with open(OUT / "registration.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(results[0]))
    w.writeheader()
    w.writerows(results)
print(f"\nwrote outputs/05a/registration.csv ({len(results)} rows)")

  direction check house_23-sketchfab: median distance under T 0.001 m, under inv(T) 63.7 m (limit for T 0.448 m)


house_23-sketchfab: 116,681 triangles, diagonal 22.4, 999,998 scan points, voxel 0.179; load 1 s, register 287 s; seed 1 picked, fail; peak RSS so far 0.32 GB


  direction check house_13-sketchfab: median distance under T 0.001 m, under inv(T) 136.8 m (limit for T 0.523 m)


house_13-sketchfab: 6,348 triangles, diagonal 26.1, 1,000,001 scan points, voxel 0.209; load 2 s, register 67 s; seed 0 picked, success; peak RSS so far 0.32 GB


  direction check house_18-sketchfab: median distance under T 0.406 m, under inv(T) 184.1 m (limit for T 0.518 m)


...

  direction check real-indoor_modelling_3-ISPRS_indoor_modelling-UVigo: median distance under T 0.042 m, under inv(T) 927.0 m (limit for T 0.986 m)


real-indoor_modelling_3-ISPRS_indoor_modelling-UVigo: 79,736 triangles, diagonal 49.3, 1,000,000 scan points, voxel 0.394; load 2 s, register 26 s; seed 3 picked, fail; peak RSS so far 0.41 GB


  direction check real-indoor_modelling_2-ISPRS_indoor_modelling-Fire_Brigade: median distance under T 0.049 m, under inv(T) 28.7 m (limit for T 1.102 m)


real-indoor_modelling_2-ISPRS_indoor_modelling-Fire_Brigade: 115,578 triangles, diagonal 55.1, 1,000,000 scan points, voxel 0.441; load 4 s, register 26 s; seed 1 picked, success; peak RSS so far 0.41 GB



wrote outputs/05a/registration.csv (9 rows)


## Result: the counted draw (highest ICP fitness of 5 seeds)

Columns: ground-truth scale, rotation error, translation error (the **centroid error**; `|dt|` is context only), the
registration residual (median scan-to-model distance after registration, and the same under the ground truth `T`, the `resid at T` column, which is the best any registration can do on that sample: where the scan covers more than the model it is not near 0),
and whether the sample succeeded under the gate (rotation <= 2 deg, centroid <= 5 cm, and for scaled samples scale
error <= 1%). The scale error column is the estimate against the truth, so it is 0 for samples run at the prior of 1.

In [7]:
def table(rows, seed0=False):
    rot, cen, ok = ("seed0_rot_deg", "seed0_centroid_m", "seed0_success") if seed0 else ("rot_deg", "centroid_m", "success")
    head = f"{'sample':62s}{'category':18s}{'GT':>7s}{'rot err':>9s}{'centroid':>10s}"
    unit = f"{'':62s}{'':18s}{'scale':>7s}{'(deg)':>9s}{'err (m)':>10s}"
    if not seed0:
        head += f"{'|dt|':>9s}{'resid med':>11s}{'resid at T':>11s}{'scale err':>10s}"
        unit += f"{'(m)':>9s}{'(m)':>11s}{'(m)':>11s}{'(%)':>10s}"
    print(head + "  result\n" + unit)
    for m in rows:
        line = f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}{m['category']:18s}{m['gt_scale']:7.3f}{m[rot]:9.2f}{m[cen]:10.3f}"
        if not seed0:
            line += f"{m['dt_m']:9.2f}{m['resid_median_m']:11.3f}{m['gt_resid_median_m']:11.3f}" + (f"{m['scale_err_pct']:10.2f}" if m["scaled"] else f"{'-':>10s}")
        print(line + f"  {'ok' if m[ok] else 'FAIL'}")


table(results)

sample                                                        category               GT  rot err  centroid     |dt|  resid med resid at T scale err  result
                                                                                  scale    (deg)   err (m)      (m)        (m)        (m)       (%)
house_23-sketchfab                                            house               1.206     0.23     0.070     4.46      0.063      0.001      3.90  FAIL
house_13-sketchfab                                            house               1.000     0.02     0.002     0.02      0.001      0.001         -  ok
house_18-sketchfab                                            house               1.000     0.54     0.041     0.94      0.404      0.406         -  ok
indoor_modelling_4-UoM                                        simulated indoor    0.747     0.05     0.024     0.77      0.023      0.001      1.55  FAIL
indoor_modelling_3-UVigo                                      simulated indoor    1.

In [8]:
n_ok = sum(m["success"] for m in results)
n_seed0 = sum(m["seed0_success"] for m in results)
by_cat = {c: sum(m["success"] for m in results if m["category"] == c) for c in dict.fromkeys(m["category"] for m in results)}
by_scaled = {k: (sum(m["success"] for m in results if m["scaled"] == k), sum(m["scaled"] == k for m in results)) for k in (False, True)}
print(f"Succeeded (best of {len(SEED_SWEEP)} seeds by ICP fitness): {n_ok} of {len(results)}")
print("  per category: " + ", ".join(f"{c} {n}/3" for c, n in by_cat.items()))
print(f"  scale 1: {by_scaled[False][0]}/{by_scaled[False][1]}, scaled: {by_scaled[True][0]}/{by_scaled[True][1]}")
print(f"  context, seed 0 alone: {n_seed0} of {len(results)}")
n_no_scale = sum(succeeded(m["rot_deg"], m["centroid_m"], 0.0, False) for m in results)
print(f"  context, ignoring the scale criterion: {n_no_scale} of {len(results)}")
print(f"\nRegistration criterion (a judgement call): at least {GATE_MIN_SUCCESS} of {len(results)} succeed -> "
      f"{'PASS' if n_ok >= GATE_MIN_SUCCESS else 'FAIL'}")

# Independent check: re-read the CSV the next notebook will read and recompute success per row from its numeric columns and
# the threshold constants (not from the in-memory flags), then compare with the stored column and the count printed above.
saved = list(csv.DictReader(open(OUT / "registration.csv")))
recomputed = [float(r["rot_deg"]) <= GATE_ROT_DEG and float(r["centroid_m"]) <= GATE_TRANS_M
              and (r["scaled"] != "True" or float(r["scale_err_pct"]) <= 100 * GATE_SCALE_REL) for r in saved]
assert len(saved) == 9 and recomputed == [r["success"] == "True" for r in saved] and sum(recomputed) == n_ok

Succeeded (best of 5 seeds by ICP fitness): 4 of 9
  per category: house 2/3, simulated indoor 1/3, real indoor 1/3
  scale 1: 4/5, scaled: 0/4
  context, seed 0 alone: 2 of 9
  context, ignoring the scale criterion: 5 of 9

Registration criterion (a judgement call): at least 6 of 9 succeed -> FAIL


### Seed 0 alone (context only)

The same table for the seed declared in advance, without the ICP-fitness pick. Where it differs from the table above, the
pick mattered (or did harm, if seed 0 was the better draw).

In [9]:
table(results, seed0=True)

print(f"\nRotation error (deg) of each of the five draws, and the ICP fitness that picked one (* = picked):")
for m in results:
    errs, fits = m["seeds_rot_deg"].split(";"), m["seeds_icp_fitness"].split(";")
    cells_ = [f"{float(e):7.2f}/{float(f):.3f}{'*' if i == m['best_seed'] else ' '}" for i, (e, f) in enumerate(zip(errs, fits))]
    print(f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}" + " ".join(cells_))

sample                                                        category               GT  rot err  centroid  result
                                                                                  scale    (deg)   err (m)
house_23-sketchfab                                            house               1.206     0.23     0.070  FAIL
house_13-sketchfab                                            house               1.000     0.02     0.002  ok
house_18-sketchfab                                            house               1.000   179.95     1.883  FAIL
indoor_modelling_4-UoM                                        simulated indoor    0.747   160.07     6.878  FAIL
indoor_modelling_3-UVigo                                      simulated indoor    1.000     0.09     0.023  ok
indoor_modelling_0-TUBS1                                      simulated indoor    0.689     0.13     0.171  FAIL
real-indoor_modelling_1-TUBS2                                 real indoor         1.258     0.18     0.2

## Run summary

In [10]:
print(f"notebook wall time {time.perf_counter() - t_notebook:.0f} s; in-kernel peak RSS {peak_rss_gb():.2f} GB")
print(f"{'sample':62s}{'load (s)':>10s}{'register (s)':>14s}")
for m in results:
    print(f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}{m['load_s']:10.0f}{m['register_s']:14.0f}")

notebook wall time 631 s; in-kernel peak RSS 0.41 GB
sample                                                          load (s)  register (s)
house_23-sketchfab                                                     1           287
house_13-sketchfab                                                     2            67
house_18-sketchfab                                                     3            52
indoor_modelling_4-UoM                                                 2            28
indoor_modelling_3-UVigo                                               3            22
indoor_modelling_0-TUBS1                                               4            19
real-indoor_modelling_1-TUBS2                                          4            66
real-indoor_modelling_3-UVigo                                          2            26
real-indoor_modelling_2-Fire_Brigade                                   4            26


## Answer

Numbers below are from this notebook's run. The parameters are 02's, chosen after seeing UVigo's ground truth (one of the
nine samples), and each sample is one sample.

**Does the registration criterion pass?** No: 4 of 9 samples succeeded, against the 6 the roadmap asks for. By category:
house 2 of 3, simulated indoor 1 of 3, real indoor 1 of 3. Seed 0 alone would have given 2 of 9. Ignoring the scale
criterion altogether would give 5 of 9, which also fails. The notebook records this as the result; the roadmap's
"if 05 fails" rule (at most one week of tuning inside 0.5, else a written negative result) is a decision for later.

**Where it fails.**

- *Rotation is not the problem.* After the ICP-fitness pick, all nine samples are within 2 degrees (largest 0.54, house_18).
  The failures are translation and scale.
- *Scaled samples: 0 of 4.* The RMS-radius scale estimate was 1.5%, 3.9%, 5.3% and 7.4% off, against a 1% gate (and
  scale error is about the only thing wrong with `indoor_modelling_4-UoM`: 0.05 degrees, 2.4 cm centroid error). Its
  assumption, that the scan is a clean complete copy of the model, holds for `fur_010` (2.2% in 02) and increasingly less for
  rooms. The worst is the real indoor TUBS2 scan, at 7.4%. A scale estimate with a ground-truth flag as input is already
  a favourable setting, so this is the weakest part.
- *Centroid error is within 5 cm on 5 of 9* (house_13, house_18, `indoor_modelling_4`, `indoor_modelling_3`, Fire Brigade)
  and outside on 4: house_23 (7.0 cm), `indoor_modelling_0` (17 cm), real TUBS2 (22 cm) and real UVigo (14 cm, the same as
  02). The voxel is 0.18 to 0.44 m, so ICP at that resolution settles within a fraction of a voxel but not within 5 cm. 02's
  suggestion of a finer-voxel ICP stage is untested here.
- *Scale 1: 4 of 5 succeeded*; real UVigo missed on centroid error only (0.34 degrees, 14 cm).

**Does picking by ICP fitness work?** On these nine samples, yes: in every one the picked draw has the smallest rotation
error of the five (ties included). It matters: seed 0 is a ~180 or ~160 degree failure on house_18 and
`indoor_modelling_4`, a 116 degree failure on real UVigo, and a 5 m centroid error on Fire Brigade (0.84 degrees). House_18
and Fire Brigade succeed only because of the pick. That is nine samples and one rule, so it is a hypothesis, not a result.
The cost is five RANSAC runs per sample (features are computed once; 8 s to 3 minutes per sample across runs, depending on the load on the machine).

**Reading the residuals.** The registration residual is a quality measure, not an accuracy level. Under the ground truth
itself it is 0.001 m on the simulated and house samples that have a complete model, and 0.034 to 0.049 m on the real
scans (extra clutter and areas outside the model); house_18's 0.40 m is the same under `T` (0.406), because the scan
covers much more than its model (ICP fitness 0.47), so that number says nothing about the registration. For the same reason
the direction check on the ground truth (2% of the diagonal) passes house_18 at 0.406 m against a ~0.52 m limit: it separates
`T` from `inv(T)` (off by orders of magnitude), and says nothing about fine accuracy.

**Limits.** One run of each sample (RANSAC seeds are fixed, but Open3D is multithreaded, so exact repeatability is not guaranteed. Checked once: a second full run reproduced every verdict, the picked seeds and all numbers except house_13's, which moved by about 1e-4 degrees and 6e-6 m, far below any threshold); the scan is a 1M-point
subsample; the scale handling uses the benchmark's scaled / unscaled flag, which a real run does not have; the thresholds
are judgement calls, not from a standard. Peak memory was low (see the run summary), so the nine samples would fit a richer
pipeline on this machine.